# InceptionV3 — TensorFlow / Keras

Factorised inception modules with 3-A / 4-B / 2-C grid blocks. Input: **299×299**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import roc_auc_score
print("TF:", tf.__version__)


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

def _conv_bn(x, filters, kernel_size, strides=1, padding="valid", activation="relu"):
    x = layers.Conv2D(filters, kernel_size, strides=strides, padding=padding, use_bias=False)(x)
    x = layers.BatchNormalization(scale=False)(x)
    if activation:
        x = layers.Activation(activation)(x)
    return x

def _inception_a(x, pool_filters):
    """35x35 Inception module (type A). Factorises 5x5 into two 3x3."""
    b0 = _conv_bn(x, 64, 1)
    b1 = _conv_bn(x, 48, 1)
    b1 = _conv_bn(b1, 64, 5, padding="same")
    b2 = _conv_bn(x, 64, 1)
    b2 = _conv_bn(b2, 96, 3, padding="same")
    b2 = _conv_bn(b2, 96, 3, padding="same")
    b3 = layers.AveragePooling2D(3, strides=1, padding="same")(x)
    b3 = _conv_bn(b3, pool_filters, 1)
    return layers.Concatenate()([b0, b1, b2, b3])

def _reduction_a(x):
    """35x35 -> 17x17, 288 -> 768 channels."""
    b0 = _conv_bn(x, 384, 3, strides=2)
    b1 = _conv_bn(x, 64, 1)
    b1 = _conv_bn(b1, 96, 3, padding="same")
    b1 = _conv_bn(b1, 96, 3, strides=2)
    b2 = layers.MaxPooling2D(3, strides=2)(x)
    return layers.Concatenate()([b0, b1, b2])

def _inception_b(x, branch_filters):
    """17x17 Inception module (type B). Factorises nxn into 1xn + nx1."""
    b0 = _conv_bn(x, 192, 1)
    b1 = _conv_bn(x, branch_filters, 1)
    b1 = _conv_bn(b1, branch_filters, (1, 7), padding="same")
    b1 = _conv_bn(b1, 192, (7, 1), padding="same")
    b2 = _conv_bn(x, branch_filters, 1)
    b2 = _conv_bn(b2, branch_filters, (7, 1), padding="same")
    b2 = _conv_bn(b2, branch_filters, (1, 7), padding="same")
    b2 = _conv_bn(b2, branch_filters, (7, 1), padding="same")
    b2 = _conv_bn(b2, 192, (1, 7), padding="same")
    b3 = layers.AveragePooling2D(3, strides=1, padding="same")(x)
    b3 = _conv_bn(b3, 192, 1)
    return layers.Concatenate()([b0, b1, b2, b3])

def _reduction_b(x):
    """17x17 -> 8x8, 768 -> 1280 channels."""
    b0 = _conv_bn(x, 192, 1)
    b0 = _conv_bn(b0, 320, 3, strides=2)
    b1 = _conv_bn(x, 192, 1)
    b1 = _conv_bn(b1, 192, (1, 7), padding="same")
    b1 = _conv_bn(b1, 192, (7, 1), padding="same")
    b1 = _conv_bn(b1, 192, 3, strides=2)
    b2 = layers.MaxPooling2D(3, strides=2)(x)
    return layers.Concatenate()([b0, b1, b2])

def _inception_c(x):
    """8x8 Inception module (type C). Parallel 1x3 and 3x1 branches."""
    b0 = _conv_bn(x, 320, 1)
    b1 = _conv_bn(x, 384, 1)
    b1a = _conv_bn(b1, 384, (1, 3), padding="same")
    b1b = _conv_bn(b1, 384, (3, 1), padding="same")
    b1 = layers.Concatenate()([b1a, b1b])
    b2 = _conv_bn(x, 448, 1)
    b2 = _conv_bn(b2, 384, 3, padding="same")
    b2a = _conv_bn(b2, 384, (1, 3), padding="same")
    b2b = _conv_bn(b2, 384, (3, 1), padding="same")
    b2 = layers.Concatenate()([b2a, b2b])
    b3 = layers.AveragePooling2D(3, strides=1, padding="same")(x)
    b3 = _conv_bn(b3, 192, 1)
    return layers.Concatenate()([b0, b1, b2, b3])

def build_inceptionv3(num_classes=1000, input_shape=(299, 299, 3)):
    inputs = keras.Input(shape=input_shape)

    # Stem: 299 -> 149 -> 147 -> 73 -> 71 -> 35
    x = _conv_bn(inputs, 32, 3, strides=2)          # 149x149
    x = _conv_bn(x, 32, 3)                           # 147x147
    x = _conv_bn(x, 64, 3, padding="same")           # 147x147
    x = layers.MaxPooling2D(3, strides=2)(x)         # 73x73
    x = _conv_bn(x, 80, 1)                           # 73x73
    x = _conv_bn(x, 192, 3)                          # 71x71
    x = layers.MaxPooling2D(3, strides=2)(x)         # 35x35

    # Inception-A x3: 35x35 (pool_filters vary: 32, 64, 64)
    x = _inception_a(x, pool_filters=32)             # 35x35x256
    x = _inception_a(x, pool_filters=64)             # 35x35x288
    x = _inception_a(x, pool_filters=64)             # 35x35x288

    # Grid Reduction-A: 35 -> 17
    x = _reduction_a(x)                              # 17x17x768

    # Inception-B x4: 17x17 (branch_filters: 128, 160, 160, 192)
    x = _inception_b(x, branch_filters=128)          # 17x17x768
    x = _inception_b(x, branch_filters=160)          # 17x17x768
    x = _inception_b(x, branch_filters=160)          # 17x17x768
    x = _inception_b(x, branch_filters=192)          # 17x17x768

    # Grid Reduction-B: 17 -> 8
    x = _reduction_b(x)                              # 8x8x1280

    # Inception-C x2: 8x8
    x = _inception_c(x)                              # 8x8x2048
    x = _inception_c(x)                              # 8x8x2048

    # Head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(num_classes, activation="softmax")(x)

    return keras.Model(inputs, x, name="InceptionV3")

In [ ]:
model = build_inceptionv3(num_classes=10)
model.summary()


In [ ]:
dummy = tf.random.normal([2, 299, 299, 3])
out = model(dummy, training=False)
print("Output shape:", out.shape)


In [ ]:
total = sum(p.numpy().size for p in model.trainable_weights)
print(f"Trainable params: {total:,}")


In [ ]:
layer_info = [(l.name, l.__class__.__name__, l.count_params()) for l in model.layers]
for name, typ, params in layer_info[-20:]:
    print(f"{name:45s}  {typ:30s}  {params:>10,}")


## Training

Use `ImageDataGenerator` with augmentation for from-scratch training.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

BATCH = 16
IMG_SIZE = (299, 299)
TRAIN_DIR = "dataset/train"
VAL_DIR   = "dataset/val"

train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
)
val_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH, class_mode="categorical")
val_data   = val_gen.flow_from_directory(VAL_DIR,   target_size=IMG_SIZE, batch_size=BATCH, class_mode="categorical", shuffle=False)

NUM_CLASSES = len(train_data.class_indices)
print("Classes:", NUM_CLASSES)


In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau

model = build_inceptionv3(num_classes=NUM_CLASSES)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks = [
    ModelCheckpoint("inceptionv3_best.keras", monitor="val_accuracy", save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.1, patience=5, min_lr=1e-7, verbose=1),
]

history = model.fit(train_data, validation_data=val_data, epochs=30, callbacks=callbacks)


## Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["accuracy"], label="train"); axes[0].plot(history.history["val_accuracy"], label="val")
axes[0].set_title("Accuracy"); axes[0].legend()
axes[1].plot(history.history["loss"], label="train"); axes[1].plot(history.history["val_loss"], label="val")
axes[1].set_title("Loss"); axes[1].legend()
plt.tight_layout(); plt.show()


## Inference

In [ ]:
from tensorflow.keras.preprocessing.image import load_img, img_to_array

CLASS_NAMES = list(train_data.class_indices.keys())

def predict_image(img_path, model, class_names, img_size=(299, 299)):
    img = load_img(img_path, target_size=img_size)
    x = img_to_array(img) / 255.0
    x = np.expand_dims(x, 0)
    probs = model.predict(x, verbose=0)[0]
    idx = np.argmax(probs)
    print(f"Prediction: {class_names[idx]}  ({probs[idx]*100:.1f}%)")
    return class_names[idx], probs

# predict_image("test.jpg", model, CLASS_NAMES)


## ROC-AUC

In [ ]:
val_gen.reset()
y_true = val_data.classes
y_prob = model.predict(val_data, verbose=1)


In [ ]:
from sklearn.preprocessing import label_binarize

y_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
auc   = roc_auc_score(y_bin, y_prob, average="macro", multi_class="ovr")
print(f"Macro ROC-AUC: {auc:.4f}")

from sklearn.metrics import roc_curve, auc as auc_score
plt.figure(figsize=(8, 6))
for i in range(min(NUM_CLASSES, 10)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
    plt.plot(fpr, tpr, lw=1, label=f"{CLASS_NAMES[i]} (AUC={auc_score(fpr,tpr):.2f})")
plt.plot([0,1],[0,1],"k--"); plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("ROC Curves"); plt.legend(fontsize=7); plt.show()
